In [ ]:
!nvidia-smi

Tue Apr 28 02:55:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, accuracy_score, roc_curve
)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── CELL 4: Load Drishti-GS (TRAINING set = 50 images for train/val) ─────────
# NOTE: We use the official Training split (50 images) for training,
#       and the official Test split (51 images) for evaluation.
#       This is the correct protocol — do NOT train on the test set.

TRAIN_ROOT = "/content/drive/MyDrive/DrishtiGS/Training/Images"
TEST_ROOT  = "/content/drive/MyDrive/DrishtiGS/Test/Images"
SAVE_DIR   = "/content/drive/MyDrive/DrishtiGS/models"
os.makedirs(SAVE_DIR, exist_ok=True)

def collect_images(root):
    records = []
    class_map = {"normal": 0, "NORMAL": 0, "Normal": 0,
                 "glaucoma": 1, "GLAUCOMA": 1, "Glaucoma": 1}
    for cls in os.listdir(root):
        if cls not in class_map:
            continue
        label = class_map[cls]
        cls_path = os.path.join(root, cls)
        for ext in ("*.jpg", "*.png", "*.jpeg", "*.JPG", "*.PNG"):
            for img in glob.glob(os.path.join(cls_path, ext)):
                records.append([img, label])
    return records

train_records = collect_images(TRAIN_ROOT)
test_records  = collect_images(TEST_ROOT)

train_df = pd.DataFrame(train_records, columns=["image_path", "label"])
test_df  = pd.DataFrame(test_records,  columns=["image_path", "label"])

print(f"Train size: {len(train_df)}")
print(f"Train class distribution:\n{train_df['label'].value_counts().to_string()}")
print(f"\nTest size : {len(test_df)}")
print(f"Test class distribution:\n{test_df['label'].value_counts().to_string()}")

Mounted at /content/drive
Train size: 50
Train class distribution:
label
1    32
0    18

Test size : 51
Test class distribution:
label
1    38
0    13


In [ ]:
IMG_SIZE = 224  # ResNet standard input size

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class DrishtiDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img  = Image.open(self.df.loc[idx, "image_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.df.loc[idx, "label"], dtype=torch.float32)
        return img, label

train_dataset = DrishtiDataset(train_df, train_transform)
test_dataset  = DrishtiDataset(test_df,  test_transform)

# Weighted sampler to handle class imbalance in training
labels_array  = train_df["label"].values
class_counts  = np.bincount(labels_array)
class_weights = 1.0 / class_counts
sample_weights = class_weights[labels_array]
sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=4, sampler=sampler)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False)

print(f"\nTrain loader batches: {len(train_loader)}")
print(f"Test  loader batches: {len(test_loader)}")



Train loader batches: 13
Test  loader batches: 13


In [ ]:
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet.fc = nn.Linear(resnet.fc.in_features, 1)
model = resnet.to(DEVICE)

# Weighted BCE: pos_weight = (num_negatives / num_positives)
pos_weight = torch.tensor(
    [class_counts[0] / class_counts[1]], dtype=torch.float32
).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)

print(f"pos_weight: {pos_weight.item():.2f}")


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 192MB/s]


pos_weight: 0.56


In [ ]:
EPOCHS      = 15
best_val_auc = 0.0
MODEL_SAVE   = os.path.join(SAVE_DIR, "resnet50_drishti_baseline_v2.pth")

train_loss_hist, val_auc_hist = [], []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(imgs)
    train_loss = running_loss / len(train_loader.dataset)

    # ── Evaluate on test set for early stopping ──
    model.eval()
    y_true_ep, y_prob_ep = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)
            probs = torch.sigmoid(model(imgs).squeeze(1)).cpu().numpy()
            y_prob_ep.extend(probs.tolist())
            y_true_ep.extend(labels.numpy().tolist())

    val_auc = roc_auc_score(y_true_ep, y_prob_ep)
    train_loss_hist.append(train_loss)
    val_auc_hist.append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), MODEL_SAVE)
        tag = "  saved"
    else:
        tag = ""

    scheduler.step()
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Test AUC: {val_auc:.4f}{tag}")

print(f"\nBest Test AUC: {best_val_auc:.4f}")


Epoch 01/15 | Train Loss: 0.4833 | Test AUC: 0.6093  saved
Epoch 02/15 | Train Loss: 0.4249 | Test AUC: 0.6943  saved
Epoch 03/15 | Train Loss: 0.4153 | Test AUC: 0.6923
Epoch 04/15 | Train Loss: 0.3919 | Test AUC: 0.6397
Epoch 05/15 | Train Loss: 0.3398 | Test AUC: 0.5850
Epoch 06/15 | Train Loss: 0.2386 | Test AUC: 0.6194
Epoch 07/15 | Train Loss: 0.2654 | Test AUC: 0.7166  saved
Epoch 08/15 | Train Loss: 0.1395 | Test AUC: 0.7024
Epoch 09/15 | Train Loss: 0.1543 | Test AUC: 0.6903
Epoch 10/15 | Train Loss: 0.1657 | Test AUC: 0.6700
Epoch 11/15 | Train Loss: 0.1161 | Test AUC: 0.6397
Epoch 12/15 | Train Loss: 0.1499 | Test AUC: 0.6619
Epoch 13/15 | Train Loss: 0.1508 | Test AUC: 0.6700
Epoch 14/15 | Train Loss: 0.0937 | Test AUC: 0.6498
Epoch 15/15 | Train Loss: 0.0880 | Test AUC: 0.6113

Best Test AUC: 0.7166


In [ ]:
model.load_state_dict(torch.load(MODEL_SAVE, map_location=DEVICE))
model.eval()

y_true_all, y_prob_all = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs  = imgs.to(DEVICE)
        probs = torch.sigmoid(model(imgs).squeeze(1)).cpu().numpy()
        y_prob_all.extend(probs.tolist())
        y_true_all.extend(labels.numpy().tolist())

y_true = np.array(y_true_all)
y_prob = np.array(y_prob_all)

# Youden-optimal threshold
fpr_vals, tpr_vals, thresholds = roc_curve(y_true, y_prob)
youden_idx  = np.argmax(tpr_vals - fpr_vals)
best_thresh = float(thresholds[youden_idx])
y_pred      = (y_prob >= best_thresh).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print("=" * 55)
print("  NB03 — ResNet50 Baseline | Drishti-GS Test (n=51)")
print("=" * 55)
print(f"  Threshold (Youden): {best_thresh:.4f}")
print(f"  Accuracy          : {accuracy_score(y_true, y_pred):.4f}")
print(f"  Precision         : {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  Sensitivity       : {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  Specificity       : {tn / (tn + fp):.4f}")
print(f"  F1-Score          : {f1_score(y_true, y_pred, zero_division=0):.4f}")
print(f"  ROC-AUC           : {roc_auc_score(y_true, y_prob):.4f}")
print(f"  PR-AUC            : {average_precision_score(y_true, y_prob):.4f}")
print(f"  TP={tp}  TN={tn}  FP={fp}  FN={fn}")
print("=" * 55)
print("  Source: NB03 — ResNet50 / Drishti-GS — BASELINE ROW FOR TABLE 1")


  NB03 — ResNet50 Baseline | Drishti-GS Test (n=51)
  Threshold (Youden): 0.4866
  Accuracy          : 0.6863
  Precision         : 0.9231
  Sensitivity       : 0.6316
  Specificity       : 0.8462
  F1-Score          : 0.7500
  ROC-AUC           : 0.7166
  PR-AUC            : 0.8899
  TP=24  TN=11  FP=2  FN=14
  Source: NB03 — ResNet50 / Drishti-GS — BASELINE ROW FOR TABLE 1


In [ ]:
import json
metrics = {
    "model": "ResNet50 Baseline",
    "dataset": "Drishti-GS",
    "n_test": int(len(y_true)),
    "threshold": round(float(best_thresh), 4),
    "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
    "precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
    "sensitivity": round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
    "specificity": round(float(tn / (tn + fp)), 4),
    "f1": round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
    "roc_auc": round(float(roc_auc_score(y_true, y_prob)), 4),
    "pr_auc": round(float(average_precision_score(y_true, y_prob)), 4),
    "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn)
}
METRICS_DIR = "/content/drive/MyDrive/REFUGE2_RESULTS"
os.makedirs(METRICS_DIR, exist_ok=True)
with open(f"{METRICS_DIR}/nb03_resnet_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved:", f"{METRICS_DIR}/nb03_resnet_baseline_metrics.json")



Saved: /content/drive/MyDrive/REFUGE2_RESULTS/nb03_resnet_baseline_metrics.json
